In [ ]:
import geopandas as gpd
import pandas as pd
import matplotlib.pyplot as plt

In [ ]:
plt.rcParams["font.family"] = 'NanumGothic'
PATH = '../P1_Data/03.서비스권역내(국토정보플랫폼)/'
PATH2 = '../P1_Data/04.서비스권역내(기존데이터)/지오코딩/'
path_1 = '../P1_Data/01.External/양주시 분석 기초데이터/'
path_2 = '../P1_Data/01.External/양주시 분석 기초데이터(2차)/'

In [ ]:
#행정동별 작업 격자
hjd_geo = gpd.read_file('../P1_Data/02.Pretreatment/hjd_geo2.gpkg')
#격자불러오기
layer = gpd.read_file('../P1_Data/02.Pretreatment/격자/total_pop_grid100.gpkg')  #총인구 100m격자
old_grid_100 = gpd.read_file('../P1_Data/02.Pretreatment/격자/old_grid100.gpkg')  #고령인구 100m 격자
young_grid_100 = gpd.read_file('../P1_Data/02.Pretreatment/격자/young_grid100.gpkg')  #유아인구 100m 격자
studend_grid_100 = gpd.read_file('../P1_Data/02.Pretreatment/격자/studend_grid100.gpkg')  #학령인구 100m 격자

In [ ]:
#권역별 작업 격자
rei_geo = gpd.read_file('../P1_Data/02.Pretreatment/result_rei_boudary.gpkg')
#격자불러오기
layer_rei = gpd.read_file('../P1_Data/02.Pretreatment/격자/rei_total_pop_grid100.gpkg') #총인구 100m격자
old_grid_rei = gpd.read_file('../P1_Data/02.Pretreatment/격자/rei_old_grid100.gpkg') #고령인구 100m 격자
young_grid_rei = gpd.read_file('../P1_Data/02.Pretreatment/격자/rei_young_grid100.gpkg') #유아인구 100m 격자
studend_grid_rei = gpd.read_file('../P1_Data/02.Pretreatment/격자/rei_studend_grid100.gpkg') #학령인구 100m 격자

In [ ]:
#결과생성 파일
result_hjd_geo = hjd_geo.copy()
result_rei_geo = rei_geo.copy()
result_hjd_geo = result_hjd_geo[['hjd_cd','hjd_nm','geometry']]
result_rei_geo = result_rei_geo[['rei_nm','geometry']]
result_rei_geo.columns = ['권역명','geometry']

### 기존데이터 서비스권역 내 지표
- 어린이집 및 유치원 서비스권역 내  영유아인구 비율
- 초등학교 서비스권역 내 학령인구 비율
- 공공문화시설 서비스권역 내 인구 비율
- 도서관 서비스권역 내 인구 비율
- 공공체육시설 서비스권역 내 인구 비율
- 공원 서비스권역 내 인구 비율
- 병원 서비스권역 내 인구 비율
- 노인복지시설 서비스권역 내 노인인구 비율
- 사회복지시설 서비스권역 내 인구 비율
- 소방서 접근성
- 경찰서(파출소) 접근성
- 버스정류장 서비스권역 내 인구 비율

In [ ]:
def assessibility(facil, size , layer):
    # 시설 버퍼 생성
    buffer = gpd.GeoSeries(facil.buffer(size))
    facil_buffer = pd.concat([facil, buffer], axis=1)
    facil_buffer.rename(columns={0: 'buffer_geometry'}, inplace=True)
    facil_buffer_geo = gpd.GeoDataFrame(facil_buffer, geometry='buffer_geometry')
    facil_buffer_geo_2 = facil_buffer_geo.drop('geometry',axis=1)
    facil_intersection = gpd.overlay(layer, facil_buffer_geo_2, how='intersection')

    # 격자 교차면적 계산
    a = layer.loc[0, 'geometry'].area / 2  # 격자 한개 면적 계산
    facil_intersection['area'] = facil_intersection['geometry'].area
    dupli = facil_intersection[facil_intersection['gid'].duplicated(keep=False)]

    # 중복 gid sum 계산 후 전체 면적에서 빼기
    nan = dupli.groupby('gid').sum()
    nan.reset_index(inplace=True)
    nan = nan[['gid', 'area']]
    nan['ex_area'] = a - nan['area']
    del nan['area']
    merge = pd.merge(nan, dupli, on='gid')
    merge = merge[merge['area'] > merge['ex_area']]
    merge.sort_values('area', ascending=False, inplace=True)
    merge.drop_duplicates(subset='gid', keep='first', inplace=True)
    del merge['ex_area']
    facil_intersection.drop_duplicates(subset='gid', keep='first', inplace=True)
    facil_intersection = facil_intersection[facil_intersection['area'] > a]
    facil_intersection = pd.concat([facil_intersection, merge], axis=0)

    # 격자 인구 분리
    list = facil_intersection['gid'].unique()
    buffer_nan = layer[~layer['gid'].isin(list)]
    buffer_ok = layer[layer['gid'].isin(list)]

    ok = buffer_ok.groupby('hjd_cd').sum()
    ok.reset_index(inplace=True)
    ok.columns = ['hjd_cd', 'ok_val']

    nan = buffer_nan.groupby('hjd_cd').sum()
    nan.reset_index(inplace=True)
    nan.columns = ['hjd_cd', 'nan_val']

    result_a = pd.merge(ok, nan, on='hjd_cd', how='outer')
    result_a['nan_val'] = result_a['nan_val'].fillna(0).astype(float)
    result_a['접근성지수'] = ((result_a['ok_val']/(result_a['ok_val']+result_a['nan_val']))).round(2)

    return result_a, facil_intersection

In [ ]:
def rei_assessibility(facil, size , layer_rei):
    # 시설 버퍼 생성
    buffer = gpd.GeoSeries(facil.buffer(size))
    facil_buffer = pd.concat([facil, buffer], axis=1)
    facil_buffer.rename(columns={0: 'buffer_geometry'}, inplace=True)
    facil_buffer_geo = gpd.GeoDataFrame(facil_buffer, geometry='buffer_geometry')
    facil_buffer_geo_2 = facil_buffer_geo.drop('geometry',axis=1)
    facil_intersection = gpd.overlay(layer_rei, facil_buffer_geo_2, how='intersection')

    # 격자 교차면적 계산
    a = layer_rei.loc[0, 'geometry'].area / 2  # 격자 한개 면적 계산
    facil_intersection['area'] = facil_intersection['geometry'].area
    dupli = facil_intersection[facil_intersection['gid'].duplicated(keep=False)]

    # 중복 gid sum 계산 후 전체 면적에서 빼기
    nan = dupli.groupby('gid').sum()
    nan.reset_index(inplace=True)
    nan = nan[['gid', 'area']]
    nan['ex_area'] = a - nan['area']
    del nan['area']
    merge = pd.merge(nan, dupli, on='gid')
    merge = merge[merge['area'] > merge['ex_area']]
    merge.sort_values('area', ascending=False, inplace=True)
    merge.drop_duplicates(subset='gid', keep='first', inplace=True)
    del merge['ex_area']
    facil_intersection.drop_duplicates(subset='gid', keep='first', inplace=True)
    facil_intersection = facil_intersection[facil_intersection['area'] > a]
    facil_intersection = pd.concat([facil_intersection, merge], axis=0)

    # 격자 인구 분리
    list = facil_intersection['gid'].unique()
    buffer_nan = layer_rei[~layer_rei['gid'].isin(list)]
    buffer_ok = layer_rei[layer_rei['gid'].isin(list)]

    ok = buffer_ok.groupby('권역명').sum()
    ok.reset_index(inplace=True)
    ok.columns = ['권역명', 'ok_val','권역코드']

    nan = buffer_nan.groupby('권역명').sum()
    nan.reset_index(inplace=True)
    nan.columns = ['권역명', 'nan_val','권역코드']

    result_a = pd.merge(ok, nan, on='권역명', how='outer')
    result_a['nan_val'] = result_a['nan_val'].fillna(0).astype(float)
    result_a['접근성지수'] = ((result_a['ok_val']/(result_a['ok_val']+result_a['nan_val']))).round(2)

    return result_a, facil_intersection

In [ ]:
#### 어린이집 및 유치원 서비스권역 내 영유아인구 비율
child1 = gpd.read_file(PATH2+'어린이집.shp',encoding='cp949')
child2 = gpd.read_file(PATH2+'유치원.shp',encoding='cp949')
child1 = child1[['행정동코드','행정동명','geometry']]
child2 = child2[['행정동코드','행정동명','geometry']]
child = pd.concat([child1, child2], axis=0)
child0 = gpd.GeoDataFrame(child,geometry='geometry')

In [ ]:
# 행정동별 함수호출
child_accessibility = assessibility(child0,750,young_grid_100)
child_acc = child_accessibility[0] #이용접근
child_buffer = child_accessibility[1] #버퍼
# 권역별 함수호출
child_accessibility_rei = rei_assessibility(child0,750,young_grid_rei)
child_acc_rei = child_accessibility_rei[0] #이용접근
child_buffer_rei = child_accessibility_rei[1] #버퍼

In [ ]:
#행정동별 후작업
result = child_acc[['hjd_cd','접근성지수']]
result.columns = ['hjd_cd','어린이집 및 유치원 서비스권역 내  영유아인구 비율(750m)']
result_hjd_geo = pd.merge(result_hjd_geo,result,on='hjd_cd')

#권역별 후작업
result_r = child_acc_rei[['권역명','접근성지수']]
result_r.columns = ['권역명','어린이집 및 유치원 서비스권역 내  영유아인구 비율(750m)']
result_rei_geo = pd.merge(result_rei_geo,result_r,on='권역명')

In [ ]:
### 초등학교
elechild = gpd.read_file(PATH2+'초등학교.shp', encoding='cp949')

In [ ]:
#행정동별함수호출
elechild_accessibility = assessibility(elechild,750,studend_grid_100)
elechild_acc = elechild_accessibility[0] #이용접근
elechild_buffer = elechild_accessibility[1] #버퍼
# 권역별 함수호출
elechild_accessibility_rei = rei_assessibility(child0,750,studend_grid_rei)
elechild_acc_rei = elechild_accessibility_rei[0] #이용접근
elechild_buffer_rei = elechild_accessibility_rei[1] #버퍼

In [ ]:
#행정동별 후작업
result = elechild_acc[['hjd_cd','접근성지수']]
result.columns = ['hjd_cd','초등학교 서비스권역 내 학령인구 비율(750m)']
result_hjd_geo = pd.merge(result_hjd_geo,result,on='hjd_cd')

#권역별 후작업
result_r = elechild_acc_rei[['권역명','접근성지수']]
result_r.columns = ['권역명','초등학교 서비스권역 내 학령인구 비율(750m)']
result_rei_geo = pd.merge(result_rei_geo,result_r,on='권역명')

### 문화시설

In [ ]:
#### 공공문화시설 (박물관,미술관,도서관)
culture1 = gpd.read_file(PATH2+'박물관미술관.shp',encoding='cp949') #박물관 미술관
culture2 = gpd.read_file(PATH2+'도서관.shp',encoding='cp949') #도서관
culture1 = culture1[['행정동명','geometry']]
culture2 = culture2[['행정동명','geometry']]
culture = pd.concat([culture1, culture2], axis=0)
culture0 = gpd.GeoDataFrame(culture,geometry='geometry')

In [ ]:
# 행정동별 함수호출
culture_accessibility = assessibility(culture0,10000,layer)
culture_acc = culture_accessibility[0] #이용접근
culture_buffer = culture_accessibility[1] #버퍼

# 권역별 함수호출
culture_accessibility_rei = rei_assessibility(culture0,10000,layer_rei)
culture_acc_rei = culture_accessibility_rei[0] #이용접근
culture_buffer_rei = culture_accessibility_rei[1] #버퍼

In [ ]:
#후작업
result = culture_acc[['hjd_cd','접근성지수']]
result.columns = ['hjd_cd','문화시설 서비스권역 내 인구비율(10km)']
result_hjd_geo = pd.merge(result_hjd_geo,result,on='hjd_cd')

#권역별 후작업
result_r = culture_acc_rei[['권역명','접근성지수']]
result_r.columns = ['권역명','문화시설 서비스권역 내 인구비율(10km)']
result_rei_geo = pd.merge(result_rei_geo,result_r,on='권역명')

### 도서관

In [ ]:
library = gpd.read_file(PATH2+'도서관.shp', encoding='cp949')
library

In [ ]:
#함수호출
library_accessibility = assessibility(library,750,layer)
library_acc = library_accessibility[0] #이용접근
library_buffer = library_accessibility[1] #버퍼

# 권역별 함수호출
library_accessibility_rei = rei_assessibility(library,750,layer_rei)
library_acc_rei = library_accessibility_rei[0] #이용접근
library_buffer_rei = library_accessibility_rei[1] #버퍼

In [ ]:
#행정동별 후작업
result = library_acc[['hjd_cd', '접근성지수']]
result.columns = ['hjd_cd', '도서관 서비스권역 내 인구 비율(750m)']
result_hjd_geo = pd.merge(result_hjd_geo, result, on='hjd_cd')

#권역별 후작업
result_r = library_acc_rei[['권역명', '접근성지수']]
result_r.columns = ['권역명', '도서관 서비스권역 내 인구 비율(750m)']
result_rei_geo = pd.merge(result_rei_geo, result_r, on='권역명')

### 체육시설

In [ ]:
gym = gpd.read_file(PATH2+'체육시설.shp', encoding='cp949')

In [ ]:
#함수호출
gym_accessibility = assessibility(gym,750,layer)
gym_acc = gym_accessibility[0] #이용접근
gym_buffer = gym_accessibility[1] #버퍼

# 권역별 함수호출
gym_accessibility_rei = rei_assessibility(gym,750,layer_rei)
gym_acc_rei = gym_accessibility_rei[0] #이용접근
gym_buffer_rei = gym_accessibility_rei[1] #버퍼

In [ ]:
#행정동별 후작업
result = gym_acc[['hjd_cd','접근성지수']]
result.columns = ['hjd_cd','체육시설 서비스권역 내 인구 비율(750m)']
result_hjd_geo = pd.merge(result_hjd_geo,result,on='hjd_cd')

#권역별 후작업
result_r = gym_acc_rei[['권역명','접근성지수']]
result_r.columns = ['권역명','체육시설 서비스권역 내 인구 비율(750m)']
result_rei_geo = pd.merge(result_rei_geo,result_r,on='권역명')

### 공원

In [ ]:
park = gpd.read_file(PATH2+'공원.shp', encoding='cp949')

In [ ]:
#함수호출
park_accessibility = assessibility(park,750,layer)
park_acc = park_accessibility[0] #이용접근
park_buffer = park_accessibility[1] #버퍼

# 권역별 함수호출
park_accessibility_rei = rei_assessibility(park,750,layer_rei)
park_acc_rei = park_accessibility_rei[0] #이용접근
park_buffer_rei = park_accessibility_rei[1] #버퍼

In [ ]:
#후작업
result = park_acc[['hjd_cd','접근성지수']]
result.columns = ['hjd_cd','공원 서비스권역 내 인구 비율(750m)']
result_hjd_geo = pd.merge(result_hjd_geo,result,on='hjd_cd')

#권역별 후작업
result_r = park_acc_rei[['권역명','접근성지수']]
result_r.columns = ['권역명','공원 서비스권역 내 인구 비율(750m)']
result_rei_geo = pd.merge(result_rei_geo,result_r,on='권역명')

### 병원

In [ ]:
hospital1 = gpd.read_file(PATH2+'병원.shp',encoding='cp949') #병원
hospital2 = gpd.read_file(PATH2+'의원.shp',encoding='cp949') #의원
hospital1 = hospital1[['행정동명','geometry']]
hospital2 = hospital2[['행정동명','geometry']]
hospital = pd.concat([hospital1, hospital2], axis=0)
hospital0 = gpd.GeoDataFrame(hospital,geometry='geometry')

In [ ]:
# 함수호출
hospital_accessibility = assessibility(hospital0,750,layer)
hospital_acc = hospital_accessibility[0] #이용접근
hospital_buffer = hospital_accessibility[1] #버퍼
# 권역별 함수호출
hospital_accessibility_rei = rei_assessibility(hospital0,750,layer_rei)
hospital_acc_rei = hospital_accessibility_rei[0] #이용접근
hospital_buffer_rei = hospital_accessibility_rei[1] #버퍼

In [ ]:
#후작업
result = hospital_acc[['hjd_cd','접근성지수']]
result.columns = ['hjd_cd','병원 서비스권역 내 인구비율(750m)']
result_hjd_geo = pd.merge(result_hjd_geo,result,on='hjd_cd')

#권역별 후작업
result_r = hospital_acc_rei[['권역명','접근성지수']]
result_r.columns = ['권역명','병원 서비스권역 내 인구비율(750m)']
result_rei_geo = pd.merge(result_rei_geo,result_r,on='권역명')

### 노인복지시설(노인복지시설,경로당)

In [ ]:
noin1 = gpd.read_file(PATH2+'노인복지시설.shp',encoding='cp949') #노인복지시설
noin2 = gpd.read_file(PATH2+'경로당.shp',encoding='cp949') #경로당
noin1 = noin1[['행정동명','geometry']]
noin2 = noin2[['행정동명','geometry']]
noin = pd.concat([noin1, noin2], axis=0)
noin0 = gpd.GeoDataFrame(noin,geometry='geometry')

In [ ]:
# 함수호출
noin_accessibility = assessibility(noin0,750,old_grid_100)
noin_acc = noin_accessibility[0] #이용접근
noin_buffer = noin_accessibility[1] #버퍼

# 권역별 함수호출
noin_accessibility_rei = rei_assessibility(noin0,750,old_grid_rei)
noin_acc_rei = noin_accessibility_rei[0] #이용접근
noin_buffer_rei = noin_accessibility_rei[1] #버퍼

In [ ]:
#후작업
result = noin_acc[['hjd_cd','접근성지수']]
result.columns = ['hjd_cd','노인복지시설 서비스권역 내 인구비율(750m)']
result_hjd_geo = pd.merge(result_hjd_geo,result,on='hjd_cd')

#권역별 후작업
result_r = noin_acc_rei[['권역명','접근성지수']]
result_r.columns = ['권역명','노인복지시설 서비스권역 내 인구비율(750m)']
result_rei_geo = pd.merge(result_rei_geo,result_r,on='권역명')

### 사회복지시설(노인복지시설,경로당,여성복지시설)

In [ ]:
saha1 = gpd.read_file(PATH2+'노인복지시설.shp',encoding='cp949') #노인복지시설
saha2 = gpd.read_file(PATH2+'경로당.shp',encoding='cp949') #경로당
saha3 = gpd.read_file(PATH2+'여성복지시설.shp',encoding='cp949') # 여성복지시설
saha1 = saha1[['행정동명','geometry']]
saha2 = saha2[['행정동명','geometry']]
saha3 = saha3[['행정동명','geometry']]
saha = pd.concat([saha1,saha2,saha3], axis=0)
saha0 = gpd.GeoDataFrame(saha,geometry='geometry')

In [ ]:
# 함수호출
saha_accessibility = assessibility(saha0,750,layer)
saha_acc = saha_accessibility[0] #이용접근
saha_buffer = saha_accessibility[1] #버퍼

# 권역별 함수호출
saha_accessibility_rei = rei_assessibility(saha0,750,layer_rei)
saha_acc_rei = saha_accessibility_rei[0] #이용접근
saha_buffer_rei = saha_accessibility_rei[1] #버퍼

In [ ]:
#후작업
result = saha_acc[['hjd_cd','접근성지수']]
result.columns = ['hjd_cd','사회복지시설 서비스권역 내 인구비율(750m)']
result_hjd_geo = pd.merge(result_hjd_geo,result,on='hjd_cd')

#권역별 후작업
result_r = saha_acc_rei[['권역명','접근성지수']]
result_r.columns = ['권역명','사회복지시설 서비스권역 내 인구비율(750m)']
result_rei_geo = pd.merge(result_rei_geo,result_r,on='권역명')

### 소방서

In [ ]:
### 소방서
fire = gpd.read_file(PATH2+'소방서.shp', encoding='cp949')

In [ ]:
#함수호출
fire_accessibility = assessibility(fire,7500,layer)
fire_acc = fire_accessibility[0] #이용접근
fire_buffer = fire_accessibility[1] #버퍼

# 권역별 함수호출
fire_accessibility_rei = rei_assessibility(fire,7500,layer_rei)
fire_acc_rei = fire_accessibility_rei[0] #이용접근
fire_buffer_rei = fire_accessibility_rei[1] #버퍼

In [ ]:
#후작업
result = fire_acc[['hjd_cd','접근성지수']]
result.columns = ['hjd_cd','소방서 서비스권역 내 인구 비율(7.5km)']
result_hjd_geo = pd.merge(result_hjd_geo,result,on='hjd_cd')

#권역별 후작업
result_r = fire_acc_rei[['권역명','접근성지수']]
result_r.columns = ['권역명','소방서 서비스권역 내 인구 비율(7.5km)']
result_rei_geo = pd.merge(result_rei_geo,result_r,on='권역명')

### 경찰서(파출소 및 지구대)

In [ ]:
### 경찰서(파출소)
police = gpd.read_file(PATH2+'파출소및지구대.shp', encoding='cp949')

In [ ]:
#함수호출
police_accessibility = assessibility(police,7500,layer)
police_acc = police_accessibility[0] #이용접근
police_buffer = police_accessibility[1] #버퍼
# 권역별 함수호출
police_accessibility_rei = rei_assessibility(police,7500,layer_rei)
police_acc_rei = police_accessibility_rei[0] #이용접근
police_buffer_rei = police_accessibility_rei[1] #버퍼

In [ ]:
#후작업
result = police_acc[['hjd_cd','접근성지수']]
result.columns = ['hjd_cd','경찰서 서비스권역 내 인구 비율(7.5km)']
result_hjd_geo = pd.merge(result_hjd_geo,result,on='hjd_cd')
#권역별 후작업
result_r = police_acc_rei[['권역명','접근성지수']]
result_r.columns = ['권역명','경찰서 서비스권역 내 인구 비율(7.5km)']
result_rei_geo = pd.merge(result_rei_geo,result_r,on='권역명')

### 버스정류장

In [ ]:
bus = gpd.read_file(PATH2+'버스정류장.shp',encoding='cp949')

In [ ]:
bus

In [ ]:
#함수호출
bus_accessibility = assessibility(bus,300,layer)
bus_acc = bus_accessibility[0] #이용접근
bus_buffer = bus_accessibility[1] #버퍼

# 권역별 함수호출
bus_accessibility_rei = rei_assessibility(bus,300,layer_rei)
bus_acc_rei = bus_accessibility_rei[0] #이용접근
bus_buffer_rei = bus_accessibility_rei[1] #버퍼

In [ ]:
#후작업
result = bus_acc[['hjd_cd','접근성지수']]
result.columns = ['hjd_cd','버스정류장 서비스권역 내 인구 비율(300m)']
result_hjd_geo = pd.merge(result_hjd_geo,result,on='hjd_cd')

#권역별 후작업
result_r = bus_acc_rei[['권역명','접근성지수']]
result_r.columns = ['권역명','버스정류장 서비스권역 내 인구 비율(300m)']
result_rei_geo = pd.merge(result_rei_geo,result_r,on='권역명')

In [ ]:
#결과
result_service_hjd = result_hjd_geo.copy()
result_service_rei = result_rei_geo.copy()

In [ ]:
columns = ['어린이집 및 유치원 서비스권역 내  영유아인구 비율(750m)',
       '초등학교 서비스권역 내 학령인구 비율(750m)', '문화시설 서비스권역 내 인구비율(10km)','도서관 서비스권역 내 인구 비율(750m)',
       '체육시설 서비스권역 내 인구 비율(750m)', '공원 서비스권역 내 인구 비율(750m)',
       '병원 서비스권역 내 인구비율(750m)', '노인복지시설 서비스권역 내 인구비율(750m)',
       '사회복지시설 서비스권역 내 인구비율(750m)', '소방서 서비스권역 내 인구 비율(7.5km)',
       '경찰서 서비스권역 내 인구 비율(7.5km)', '버스정류장 서비스권역 내 인구 비율(300m)' ]

In [ ]:
#널값 0으로 채운 후 % 값으로 생성
result_service_hjd = result_service_hjd.fillna(0.0)
result_service_rei = result_service_rei.fillna(0.0)
result_service_hjd[columns] = result_service_hjd[columns].apply(lambda x: x * 100)
result_service_rei[columns] = result_service_rei[columns].apply(lambda x: x * 100)

In [ ]:
result_service_hjd
result_service_hjd.to_file('../P3_Result/03.SHP/서비스권역내인구비율/result_service_hjd.gpkg', driver='GPKG')
result_service_hjd.to_excel('../P3_Result/01.CSV/서비스권역내인구비율/서비스권역내_인구비율결과(행정동별).xlsx', index=False)

In [ ]:
result_service_rei
result_service_rei.to_file('../P3_Result/03.SHP/서비스권역내인구비율/result_service_rei.gpkg', driver='GPKG')
result_service_rei.to_excel('../P3_Result/01.CSV/서비스권역내인구비율/서비스권역내_인구비율결과(권역별).xlsx', index=False)

In [ ]:
result_service_rei

In [ ]:
result_service_rei

### 국토부 제공 데이터 지표 전처리
- 응급의료시설 서비스권역 내 인구비율
- 병원 서비스 권역 내 인구비율
- 노인여가복지시설 서비스권역 내 노인인구 비율
- 고속도로 IC 접근성
- 주차장 접근성

def pre_shp(df):
    #양주시만 추출
    df = df[df['sgg_nm_e'] == 'Yangju-si']

    # 새로운 컬럼 생성
    df['hjd_nm'] = '행정기관'
    df['rei_nm'] = '권역명'

    # 읍면동 코드에 따라 권역 값 할당
    df.loc[df['emd_cd'].isin(['41630101','41630102','41630103','41630104','41630105']), 'hjd_nm'] = '양주1동'
    df.loc[df['emd_cd'].isin(['41630106','41630107','41630108','41630109']), 'hjd_nm'] = '양주2동'
    df.loc[df['emd_cd'].isin(['41630110','41630111']), 'hjd_nm'] = '회천1동'
    df.loc[df['emd_cd'].isin(['41630116','41630117']), 'hjd_nm'] = '회천2동'
    df.loc[df['emd_cd'].isin(['41630115']), 'hjd_nm'] = '회천3동'
    df.loc[df['emd_cd'].isin(['41630112','41630113','41630114']), 'hjd_nm'] = '회천4동'
    df.loc[df['emd_cd'].isin(['41630250']), 'hjd_nm'] = '백석읍'
    df.loc[df['emd_cd'].isin(['41630310']), 'hjd_nm'] = '은현면'
    df.loc[df['emd_cd'].isin(['41630320']), 'hjd_nm'] = '남면'
    df.loc[df['emd_cd'].isin(['41630330']), 'hjd_nm'] = '광적면'
    df.loc[df['emd_cd'].isin(['41630340']), 'hjd_nm'] = '장흥면'

    df.loc[df['emd_cd'].isin(['41630310','41630320']), 'rei_nm'] = '1번권역'
    df.loc[df['emd_cd'].isin(['41630101','41630102','41630103','41630104','41630105','41630106','41630107','41630108','41630109','41630110','41630111','41630116','41630117','41630115','41630112','41630113','41630114']), 'rei_nm'] = '2번권역'
    df.loc[df['emd_cd'].isin(['41630250','41630330']), 'rei_nm'] = '3번권역'
    df.loc[df['emd_cd'].isin(['41630340']), 'rei_nm'] = '4번권역'

    return df

#함수호출
pre_emergency = pre_shp(데이터프레임명)

#### 응급의료시설

In [ ]:
# 응급의료시설 - 행정구역 내 인구 중 응급의료시설로부터 도로이동거리로 10km 이내 지역에 거주하는 인구의 비율
# 측정방법 - 응급의료시설로부터 서비스 권역 이내에 위치한 격자에 거주하는 인구 수 ÷ 행정구역 내 총 거주 인구 × 100(10km)
emergency = gpd.read_file(PATH+'(B100)국토통계_통계지도-국토지표-생활과 복지-응급의료시설(응급의료시설 서비스권역 내 인구 10km)-시군구_2021/154.5 응급의료시설(시군구) 서비스권역 내 인구비율(10.0km).shp',encoding='utf-8')
#양주시만 추출
pre_emergency = emergency[emergency['sgg_nm_e'] == 'Yangju-si']
pre_emergency = gpd.GeoDataFrame(pre_emergency, geometry='geometry')
#pre_emergency.to_file('../P3_Result/03.SHP/서비스권역내인구비율/응급의료시설/pre_emergency.gpkg', driver='GPKG')

#### 노인여가복지시설

In [ ]:
# 노인여가복지시설 - 행정구역 내 노인인구 중 노인여가복지시설로부터 도로이동거리로 10km 이내 지역에 거주하는 노인인구의 비율
# 측정방법 - 노인여가복지시설로부터 서비스 권역 이내에 위치한 격자에 거주하는 노인인구 수 ÷ 행정구역 내 총 거주 노인인구 × 100
noin = gpd.read_file(PATH+'(B100)국토통계_국토정책지표-노인여가복지시설 서비스권역 내 인구 비율 10km-읍면동경계_2021/126.2 노인여가복지시설(읍면동) 서비스권역 내 고령인구비율(10.0km).shp')
#양주시만 추출
noin = noin[noin['sgg_nm_e'] == 'Yangju-si']
noin = gpd.GeoDataFrame(noin, geometry='geometry')
#noin.to_file('../P3_Result/03.SHP/서비스권역내인구비율/노인여가복지시설/noin.gpkg', driver='GPKG')

#### 고속도로 IC 접근성

In [ ]:
# 고속도로IC - 가장 가까운 IC(Interchange, 나들목)까지 도로이동거리
# 측정방법 - 격자(읍면동) 중심점으로부터 가장 가까운 IC까지 도로 이동거리
highway = gpd.read_file(PATH+'(B100)국토통계_국토정책지표-고속도로IC 접근성 -읍면동경계_2021/179.3 IC(읍면동) 접근성.shp')
#양주시만 추출
highway = highway[highway['sgg_nm_e'] == 'Yangju-si']
highway = gpd.GeoDataFrame(highway, geometry='geometry')
#highway.to_file('../P3_Result/03.SHP/서비스권역내인구비율/고속도로IC/highway.gpkg', driver='GPKG')

#### 주차장

In [ ]:
# 주차장 - 행정구역 내 인구 중 주차장으로부터 도로이동거리로 750m 이내 지역에 거주하는 인구의 비율
# 측정방법 - 주차장으로부터 서비스 권역 이내에 위치한 격자에 거주하는 인구 수 ÷ 행정구역 내 총 거주 인구 × 100
parking = gpd.read_file(PATH+'(B100)국토통계_국토정책지표-주차장 서비스권역 내 인구 비율 750m-읍면동경계_2021/178.2 주차장(읍면동) 서비스권역 내 인구비율(0.75km).shp')
#양주시만 추출
parking = parking[parking['sgg_nm_e'] == 'Yangju-si']
parking = gpd.GeoDataFrame(parking, geometry='geometry')
#parking.to_file('../P3_Result/03.SHP/서비스권역내인구비율/주차장/parking.gpkg', driver='GPKG')